# Ejemplo de monitorización con MLFlow

In [2]:
!pip install mlflow==2.22.0 # MLFlow JPC
# !pip install mlflow==3.10.0 #Ml flow Ribera de Castilla

In [3]:
import mlflow
print(mlflow.__version__)

2.22.0


In [4]:
import mlflow.sklearn
from mlflow.models import infer_signature
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_regression
from sklearn.metrics import mean_squared_error
import numpy as np

In [5]:
# Apuntar al servidor MLflow
mlflow.set_tracking_uri("http://192.168.1.156:5000")

In [7]:
# Crear o seleccionar un experimento
mlflow.set_experiment("testRL")

2026/03/05 18:20:16 INFO mlflow.tracking.fluent: Experiment with name 'testRL' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/5', creation_time=1772731215939, experiment_id='5', last_update_time=1772731215939, lifecycle_stage='active', name='testRL', tags={}>

In [8]:
# Generar datos de ejemplo
X, y = make_regression(n_samples=100, n_features=2, noise=0.1)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [9]:
# Iniciar experimento
with mlflow.start_run() as run:

    # Entrenar modelo
    model = LinearRegression()
    model.fit(X_train, y_train)

    # Evaluar modelo
    predictions = model.predict(X_test)
    mse = mean_squared_error(y_test, predictions)
    rmse = np.sqrt(mse)

    # Loguear parámetros
    mlflow.log_param("model_type", "LinearRegression")
    mlflow.log_param("test_size", 0.2)

    # Loguear métricas
    mlflow.log_metric("mse", mse)
    mlflow.log_metric("rmse", rmse)
    
        # Inferir la signature (usa X_train o X_test; cualquiera consistente)
    signature = infer_signature(X_train, model.predict(X_train))

    # Loguear el modelo con signature e input_example
    mlflow.sklearn.log_model(
        model,
        "model",
        input_example=X_train[:5],
        signature=signature,
    )
    
    # Verificar URI antes de guardar el modelo
    print("Artifact URI:", run.info.artifact_uri)

    # Loguear el modelo
    mlflow.sklearn.log_model(model, "model")

    print(f"MSE: {mse:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"Run ID: {mlflow.active_run().info.run_id}")

Artifact URI: mlflow-artifacts:/5/0143242d7c8b455da319987be9556761/artifacts


2026/03/05 18:20:46 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


MSE: 0.0088
RMSE: 0.0937
Run ID: 0143242d7c8b455da319987be9556761
🏃 View run bedecked-ox-600 at: http://192.168.1.156:5000/#/experiments/5/runs/0143242d7c8b455da319987be9556761
🧪 View experiment at: http://192.168.1.156:5000/#/experiments/5
